# Notebook 11: Dataset & DataLoader



## 1. Dataset vs DataLoader

**Dataset:**
A PyTorch `Dataset` is an object that holds your data. It acts as a map: "Give me index $i$, and I will give you the $i$-th sample and its label."

**DataLoader:**
A `DataLoader` is a wrapper around a Dataset. It handles:
- **Batching:** Grouping samples into mini-batches (e.g., 32 samples per batch).
- **Shuffling:** Randomizing the order of data every epoch to prevent the model from learning the order of samples.
- **Multiprocessing:** Loading data in parallel using multiple CPU cores.

In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# 1. Create raw data
X = torch.randn(1000, 20) # 1000 samples, 20 features
y = torch.randint(0, 2, (1000, 1)).float()

# 2. Create TensorDataset
# TensorDataset simply wraps tensors into a dataset object
dataset = TensorDataset(X, y)

# 3. Create DataLoader
# batch_size=32 means we update weights every 32 samples
# shuffle=True ensures the model doesn't see data in the same order every time
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Let's inspect one batch
for batch_X, batch_y in loader:
    print(f"Batch X shape: {batch_X.shape}") # [32, 20]
    print(f"Batch y shape: {batch_y.shape}") # [32, 1]
    break

Batch X shape: torch.Size([32, 20])
Batch y shape: torch.Size([32, 1])


## 2. Creating a Custom Dataset

Sometimes data isn't just in tensors (e.g., it's in CSV files or images on a disk). In these cases, we create a **Custom Dataset** by inheriting from `torch.utils.data.Dataset`.

You must implement two methods:
1. `__len__()`: Returns the total number of samples.
2. `__getitem__(idx)`: Returns the sample and label at the given index.

In [2]:
from torch.utils.data import Dataset

class MyCustomDataset(Dataset):
    def __init__(self, data_list, labels_list):
        self.data = data_list
        self.labels = labels_list
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        # Here you could load an image from a path or process a text file
        sample = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return sample, label

# Dummy data
data = [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]]
labels = [0, 1, 0, 1, 0]

custom_ds = MyCustomDataset(data, labels)
loader_custom = DataLoader(custom_ds, batch_size=2, shuffle=True)

for b_x, b_y in loader_custom:
    print("Batch X:\n", b_x)
    print("Batch y:\n", b_y)
    break

Batch X:
 tensor([[1., 2.],
        [5., 6.]])
Batch y:
 tensor([0., 0.])


## 3. Why is DataLoader important for large data?

**The Memory Problem:**
If you have a 100GB dataset, you cannot load it into RAM. `DataLoader` solves this by loading only the current mini-batch into memory.

**The Training Stability Problem:**
Full batch training is slow and can get stuck in local minima. Mini-batching adds a bit of "noise" to the gradient updates, which often helps the model find a better global minimum.